# Personal VoxCPM2 Voice Clone Backend
Run this notebook on a Colab T4 runtime. It starts a temporary personal-use API for the lightweight Streamlit app.

## 1. Select GPU
Use **Runtime → Change runtime type → T4 GPU**. Colab may not offer a T4 on every free account or at every time.

In [ ]:
!nvidia-smi

## 2. Upload the private repository ZIP
Download the private repository ZIP from GitHub first, then upload it here.

In [ ]:
from google.colab import files
import os, zipfile
uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content')
print('Project folders:', [x for x in os.listdir('/content') if 'be-by' in x])

In [ ]:
%cd /content/be-by-swe010-oct26-m4-copy
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install -r requirements-gpu.txt
!python -c "import torch; print('CUDA:', torch.cuda.is_available())"

## 3. Create a temporary access token
Keep this token private. Paste the same token into the Streamlit app's remote backend token field.

In [ ]:
import secrets, os
os.environ['VOXCPM_TOKEN'] = secrets.token_urlsafe(24)
print('VOXCPM_TOKEN =', os.environ['VOXCPM_TOKEN'])

## 4. Start the VoxCPM2 API
The first request downloads/loads the model and can take several minutes.

In [ ]:
!nohup uvicorn voice_clone_server:app --host 0.0.0.0 --port 7860 > /tmp/voxcpm_server.log 2>&1 &
!sleep 5
!curl -s http://127.0.0.1:7860/

## 5. Create a temporary public URL
Keep the Colab tab active. The URL expires when the runtime stops.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
!nohup /content/cloudflared tunnel --url http://127.0.0.1:7860 > /tmp/cloudflared.log 2>&1 &
!sleep 5
!grep -o 'https://[-a-z0-9]*\.trycloudflare.com' /tmp/cloudflared.log | head -1

Copy the `https://...trycloudflare.com` URL into the Streamlit app. Paste the token into the app's remote backend token field. The URL expires when Colab stops. Use only an authorized voice reference and stop the runtime when finished.